# Extended Usability Evaluation
This notebook extends the pipeline of `LLM_Usability_Evaluation.ipynb` (Lubos et al., FSE 2026). The original notebook is left unchanged for replication.

Additions:
1. **Model switching**: one variable selects the model (`mllm_client.py`).
2. **Measurements without a model**: colours, contrast and motion from the pixels (`video_tools.py`).
3. **Visual design evaluation**: a second criteria set (`visual_design_criteria.json`), with a higher frame rate so the model can see animations.
4. **Animation identification**: name, describe and re-implement the animation in a short clip.

The logic lives in `extended_pipeline.py`; this notebook only calls it.

## 1. Setup
Put your API key in a file named `.env` in the repository root (copy `.env.example`). The file is git-ignored. Do not paste the key into this notebook.

In [ ]:
from mllm_client import MLLMClient
import extended_pipeline as ep
import video_tools
import json

DATA_PATH = "../data/EventHelpR/"
# DATA_PATH = "../data/KnowledgeCheckR/"

# Change this one value to switch the model.
# Names starting with "gemini" use the Gemini API (GEMINI_API_KEY).
# Any other name is sent to an OpenAI-compatible endpoint (OPENAI_API_KEY, OPENAI_BASE_URL).
MODEL = "gemini-3.8-flash"

client = MLLMClient(MODEL)

## 2. Measurements without a model (no API key needed)
Dominant colours with their WCAG contrast against the background, and the screen changes found by comparing consecutive frames.

In [ ]:
VIDEO = DATA_PATH + "participant/1_access_event/access_event.mp4"

print(video_tools.video_info(VIDEO))
print(json.dumps(video_tools.colour_report(VIDEO), indent=2))

motion = video_tools.motion_summary(VIDEO)
print({k: v for k, v in motion.items() if k != "segments"})
motion["segments"][:5]

## 3. Visual design evaluation
Same per-criterion prompting as the original, with the visual design criteria. `fps` is the number of frames per second the model looks at (the original study used the default of 1). `ground=True` adds the measurements from section 2 to the prompt.

`only_task` limits the run to one recording to keep the cost low. Remove it to evaluate all tasks.

In [ ]:
from datetime import datetime

CRITERIA = "visual"   # or "nielsen" to re-run the original heuristics with another model
GROUND = False

evaluation = ep.run_usability_evaluation(DATA_PATH, client, criteria=CRITERIA, fps=5, ground=GROUND, only_task="EHR_P2_T1")

EVALUATION_FILE = f"evaluation_{CRITERIA}{'_grounded' if GROUND else ''}_{client.model}_{datetime.now():%Y%m%d%H%M%S}.json"
ep.store_json(evaluation, DATA_PATH, EVALUATION_FILE)
evaluation[0]

## 4. Aggregation and severity ranking
Unchanged from the original method: cluster similar issues (cosine similarity 0.7), summarise each cluster, then let the model rank the issues.

In [ ]:
aggregated = ep.aggregate_issues_by_task_and_similarity(evaluation, threshold=0.7)
report = ep.finalize_aggregated_report(aggregated, client)
REPORT_FILE = EVALUATION_FILE.replace("evaluation_", "report_", 1)
ep.store_json(report, DATA_PATH, REPORT_FILE)

ranking = ep.rank_report(report, client)
ep.store_json(ranking, DATA_PATH, "ranking_" + REPORT_FILE)

for task_id, issues in ranking.items():
    print(task_id)
    for n, issue in enumerate(issues[:3], 1):
        print(f"  {n}. [{issue['severity']}] {issue['description']}
     Fix: {issue['recommendation']}")

## 5. Animation identification
Give a short clip of an animation or transition (a few seconds; record it with any screen recorder). The model names it, describes it and explains how to build it. `measured` lists what the frame comparison found, to check the model's timing against.

In [ ]:
CLIP = VIDEO          # replace with the path to your own clip

result = ep.identify_animation(CLIP, client, context=None, fps=10, start=0.5, end=3.0)

print(json.dumps(result["model"], indent=2, ensure_ascii=False))
print("Measured:", result["measured"])